In [ ]:
import os
import requests
import truststore
import concurrent.futures

truststore.inject_into_ssl()

from config import settings


class TokenSession:
    """
    Gerenciador de sessão autenticada para a API do FattureWeb.
    """

    def __init__(self):
        self.base_url = settings.FATTUREWEB_BASE_URL
        self.login_path = f'{self.base_url}/auth/login'
        self.login_payload = {
            'email': settings.FATTUREWEB_USERNAME,
            'senha': settings.FATTUREWEB_PASSWORD
        }
        self.token = None

    def login(self):
        response = requests.post(self.login_path, json=self.login_payload)
        response.raise_for_status()
        json_data = response.json()

        if json_data.get('status') != 'sucesso':
            raise Exception(f"Login failed: {json_data.get('mensagem')}")

        self.token = json_data['dados'][0]['token']

    def request(self, method: str, path: str, **kwargs):
        if not self.token:
            self.login()

        headers = kwargs.pop('headers', {})
        headers['Fatture-AuthToken'] = self.token
        kwargs['headers'] = headers

        response = requests.request(method, path, **kwargs)

        if response.status_code == 401:
            self.login()
            headers['Fatture-AuthToken'] = self.token
            kwargs['headers'] = headers
            response = requests.request(method, path, **kwargs)

        response.raise_for_status()
        return response

In [2]:
ts = TokenSession()

In [3]:
ts.login()

In [ ]:
ID_CARTEIRA = 2768  # ID_MATRIX_FACIL_B

url_clientes_carteira = os.path.join(
    settings.FATTUREWEB_BASE_URL,
    "carteiras",
    f"{ID_CARTEIRA}",
    "clientes")

skip, paginacao = 0, 100

TypeError: Session.request() got an unexpected keyword argument 'limit'

In [ ]:
ts.request()

In [ ]:
def get_clients_from_wallet(session, id_carteira) -> list[str]:
    """Obtém todos os ids de cliente de uma determinada carteira."""

    PAGE_SIZE = 180
    # SEARCH_FIELDS = ""

    url = f'{session.base_url}/carteiras/{id_carteira}/clientes'

    total_faturas = session.request(
        'GET', f'{url}&count=true'
    ).json()['dados'][0]['total']

    if total_faturas == 0:
        return []

    def fetch_batch(skip: int, limit: int) -> list[str]:
        response = session.request(
            'GET',
            f'{url}&sort=mes_referencia&order=desc&skip={skip}&limit={limit}'
        )
        return [str(f['id']) for f in response.json()['dados']]

    first_batch = fetch_batch(0, PAGE_SIZE)
    fatura_ids = list(first_batch)
    step = len(first_batch)

    if step < total_faturas and step > 0:
        with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
            futures = [
                executor.submit(fetch_batch, skip, step)
                for skip in range(step, total_faturas, step)
            ]
            for future in concurrent.futures.as_completed(futures):
                fatura_ids.extend(future.result())

    return fatura_ids